In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
import csv

### Convert Audio to chromagram

In [8]:
dir_path = '/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Audio/Vocal_Split_20'
output_path = '/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Chromagram/split_20'

os.makedirs(output_path, exist_ok = True)

for filename in os.listdir(dir_path):
    file_path = os.path.join(dir_path, filename)
    y, sr = librosa.load(file_path)
    chroma = librosa.feature.chroma_stft(y = y, sr = sr)
    csv_file_name = os.path.splitext(filename)[0] + ".csv"
    csv_file_path = os.path.join(output_path, csv_file_name)
    with open(csv_file_path, 'w', newline = '') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(chroma.T)

In [ ]:
file_path = "/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Audio/Vocal_Split_20/김창환_춘향가_이별가/김창환_춘향가_이별가_GMjo_100-120.csv"

chroma = np.loadtxt(file_path, delimiter=",")

chroma = chroma.T

plt.figure(figsize=(10, 5))
librosa.display.specshow(chroma, y_axis='chroma', x_axis='time')
plt.colorbar(label='Intensity')
plt.title('Chroma Feature (From CSV)')
plt.xlabel('Time (frames)')
plt.ylabel('Pitch Class')
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CNNChromagram(nn.Module):
    def __init__(self):
        super(CNNChromagram, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size = (3, 3), padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = (3, 3), padding = 1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = (3, 3), padding = 1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size = (2,2))

        self.fc1 = nn.Linear(128, * 3 * (20//8), 64)
        self.fc2 = nn.Linear(64, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x  = self.pool(x)

        x = self.relu(self.conv2(x))
        x= self.pool(x)

        x = self.relu(self.conv3(x))
        x = self.pool(x)

        x = x.view(x.shape[0], -1)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

input_dir = "/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Chromagram/split_20"

label_map = {"GMjo": 0, "Ujo": 1}

song_data = {}

for song_folder in os.listdir(input_dir):
    song_path = os.path.join(input_dir, song_folder)
    if not os.path.isdir(song_path):
        continue

    X_list = []
    y_list = []

    for file in os.listdir(song_path):
        if file.endswith(".csv"):
            file_path = os.path.join(song_path, file)
            X_sample = np.loadtxt(file_path, delimiter=",")  # Load properly

            if X_sample.max() > 0:
                X_sample /= X_sample.max()  # Normalize

            if "GMjo" in file:
                y_label = label_map["GMjo"]
            elif "Ujo" in file:
                y_label = label_map["Ujo"]
            else:
                continue

            X_list.append(X_sample)
            y_list.append(y_label)

    if X_list:
        X_tensor = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(1)  # Shape: (batch, 1, 12, time)
        y_tensor = torch.tensor(y_list, dtype=torch.long)
        song_data[song_folder] = (X_tensor, y_tensor)

song_names = list(song_data.keys())
num_songs = len(song_names)

loo_losses = []
loo_accuracies = []

for i in range(num_songs):
    test_song = song_names[i]
    print(f"\nLOO Cross Validation - {test_song}")

    val_X, val_y = song_data[test_song]
    val_X, val_y = val_X.to(device), val_y.to(device)

    train_X = []
    train_y = []
    for j, song in enumerate(song_names):
        if i != j:
            train_X.append(song_data[song][0])
            train_y.append(song_data[song][1])

    train_X = torch.cat(train_X, dim=0).to(device)
    train_y = torch.cat(train_y, dim=0).to(device)

    model = CNNChromagram().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(20):
        model.train()
        optimizer.zero_grad()

        outputs = model(train_X)
        loss = criterion(outputs, train_y)

        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(val_X)
        val_loss = criterion(val_output, val_y)
        pred = torch.argmax(val_output, dim=1)
        accuracy = (pred == val_y).float().mean().item()

    loo_losses.append(val_loss.item())
    loo_accuracies.append(accuracy)

    print(f"Validation Loss: {val_loss.item():.4f}, Accuracy: {accuracy:.4f}")
    print(f"Prediction: {pred.cpu().numpy()}")
    print(f"Answer: {val_y.cpu().numpy()}")

print("\nLOO-CV Results")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accuracies):.4f}")


LOO-CV Results
Average Validation Loss: nan
Average Validation Accuracy: nan


### Chroma Input

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CNNChromagram(nn.Module):
    def __init__(self):
        super(CNNChromagram, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size = (3, 3), padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = (3, 3), padding = 1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = (3, 3), padding = 1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size = (2,2))

        self.fc1 = nn.Linear(128, * 3 * (20//8), 64)
        self.fc2 = nn.Linear(64, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x  = self.pool(x)

        x = self.relu(self.conv2(x))
        x= self.pool(x)

        x = self.relu(self.conv3(x))
        x = self.pool(x)

        x = x.view(x.shape[0], -1)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

input_dir = "/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Chromagram/split_20"

label_map = {"GMjo": 0, "Ujo": 1}

song_data = {}

for song_folder in os.listdir(input_dir):
    song_path = os.path.join(input_dir, song_folder)
    if not os.path.isdir(song_path):
        continue

    X_list = []
    y_list = []

    for file in os.listdir(song_path):
        if file.endswith(".csv"):
            file_path = os.path.join(song_path, file)
            X_sample = np.loadtxt(file_path, delimiter=",")  # Load properly

            if X_sample.max() > 0:
                X_sample /= X_sample.max()  # Normalize

            if "GMjo" in file:
                y_label = label_map["GMjo"]
            elif "Ujo" in file:
                y_label = label_map["Ujo"]
            else:
                continue

            X_list.append(X_sample)
            y_list.append(y_label)

    if X_list:
        X_tensor = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(1)  # Shape: (batch, 1, 12, time)
        y_tensor = torch.tensor(y_list, dtype=torch.long)
        song_data[song_folder] = (X_tensor, y_tensor)

song_names = list(song_data.keys())
num_songs = len(song_names)

loo_losses = []
loo_accuracies = []

for i in range(num_songs):
    test_song = song_names[i]
    print(f"\nLOO Cross Validation - {test_song}")

    val_X, val_y = song_data[test_song]
    val_X, val_y = val_X.to(device), val_y.to(device)

    train_X = []
    train_y = []
    for j, song in enumerate(song_names):
        if i != j:
            train_X.append(song_data[song][0])
            train_y.append(song_data[song][1])

    train_X = torch.cat(train_X, dim=0).to(device)
    train_y = torch.cat(train_y, dim=0).to(device)

    model = CNNChromagram().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(20):
        model.train()
        optimizer.zero_grad()

        outputs = model(train_X)
        loss = criterion(outputs, train_y)

        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(val_X)
        val_loss = criterion(val_output, val_y)
        pred = torch.argmax(val_output, dim=1)
        accuracy = (pred == val_y).float().mean().item()

    loo_losses.append(val_loss.item())
    loo_accuracies.append(accuracy)

    print(f"Validation Loss: {val_loss.item():.4f}, Accuracy: {accuracy:.4f}")
    print(f"Prediction: {pred.cpu().numpy()}")
    print(f"Answer: {val_y.cpu().numpy()}")

print("\nLOO-CV Results")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accuracies):.4f}")